# 02 — 候选音型窗口提取

**BWV861 音乐形式化分析系统** · Step 2

从声部分层事件中提取单声部窗口 $\omega_{i,L}^{(\lambda)}$ 和多声部块窗口 $\omega_i^{(\Lambda')}$，应用 L/D/B/tie 约束筛选。

实现文档 Section: **局部音乐事件序列**

## 2.1 约束参数

In [1]:
import sys, os
PROJECT_ROOT = r"e:\大三\Cline Projects\bien_project"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from music_analysis import config
from music_analysis.events import load_events
from music_analysis.windows import (
    extract_single_voice_windows, extract_multi_voice_windows,
    deduplicate_windows, print_window_stats, save_windows, load_windows,
)

print("约束参数:")
print(f"  L_min={config.L_MIN}, L_max={config.L_MAX}  (事件数)")
print(f"  D_min={config.D_MIN}, D_max={config.D_MAX}  (时长/QL)")
print(f"  B_max={config.B_MAX}  (小节跨度)")
print(f"  Tie 约束: 起点∉{{continue,stop}}, 终点∉{{start,continue}}")

# 加载事件数据
events, voice_layers = load_events()
print(f"\n✅ 已加载 {len(events)} 个事件, {len(voice_layers)} 个声部层")

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.

约束参数:
  L_min=3, L_max=8  (事件数)
  D_min=1.0, D_max=8.0  (时长/QL)
  B_max=2  (小节跨度)
  Tie 约束: 起点∉{continue,stop}, 终点∉{start,continue}

✅ 已加载 1456 个事件, 6 个声部层


## 2.2 单声部窗口提取

In [2]:
print("正在提取单声部候选窗口...")
single_windows = []
for (s, v), layer_events in sorted(voice_layers.items()):
    ws = extract_single_voice_windows(layer_events, (s, v))
    single_windows.extend(ws)
    if ws:
        print(f"  λ=({s}, {v}): {len(ws)} 窗口")

print(f"\n单声部窗口小计: {len(single_windows)}")

# 去重
single_windows = deduplicate_windows(single_windows, min_ratio=0.8)
print(f"去重后: {len(single_windows)} 窗口")

正在提取单声部候选窗口...
  λ=(P1-Staff1, 0): 349 窗口
  λ=(P1-Staff1, 1): 2013 窗口
  λ=(P1-Staff1, 2): 1385 窗口
  λ=(P1-Staff2, 0): 1099 窗口
  λ=(P1-Staff2, 5): 877 窗口
  λ=(P1-Staff2, 6): 658 窗口

单声部窗口小计: 6381
去重后: 1025 窗口


## 2.3 多声部窗口提取 (|Λ'|=2)

In [3]:
print("正在提取多声部候选窗口 (|Λ'|=2)...")
# 仅对事件数较多的声部层做多声部对组合，减少计算量
major_layers = {k: v for k, v in voice_layers.items() if len(v) >= 50}
print(f"  参与计算的声部层: {len(major_layers)} / {len(voice_layers)}")

multi_windows = extract_multi_voice_windows(major_layers)
print(f"多声部窗口小计: {len(multi_windows)}")

multi_windows = deduplicate_windows(multi_windows, min_ratio=0.8)
print(f"去重后: {len(multi_windows)} 窗口")

正在提取多声部候选窗口 (|Λ'|=2)...
  参与计算的声部层: 6 / 6
多声部窗口小计: 72915
去重后: 708 窗口


## 2.4 统计汇总与持久化

In [4]:
print_window_stats(single_windows, multi_windows, voice_layers)
save_windows(single_windows, multi_windows)

# 预览前 3 个单声部窗口
print("\n--- 前 3 个单声部窗口预览 ---")
for w in single_windows[:3]:
    pitches = [str(e["p_n"]) for e in w["events"]]
    print(f"  λ={w['lambda']}, L={w['L']}, D={w['D']:.1f}QL, B={w['B']}, "
          f"t=[{w['time_interval'][0]:.1f}, {w['time_interval'][1]:.1f}]")
    print(f"    音高: [{', '.join(pitches[:8])}{', ...' if len(pitches) > 8 else ''}]")

📊 候选音型窗口统计
  单声部窗口 |Ω|        = 1025
  多声部窗口 |Ω_multi|  = 708
  总候选窗口 |Ω*|        = 1733
------------------------------------------------------------
  λ=(P1-Staff1,    0): 68 单声部窗口
  λ=(P1-Staff1,    1): 276 单声部窗口
  λ=(P1-Staff1,    2): 212 单声部窗口
  λ=(P1-Staff2,    0): 189 单声部窗口
  λ=(P1-Staff2,    5): 157 单声部窗口
  λ=(P1-Staff2,    6): 123 单声部窗口

  单声部窗口 - L 范围: [3, 8], D 范围: [1.0, 8.0], B 范围: [1, 2]

  多声部窗口声部对组合:
    (P1-Staff1,1) × (P1-Staff1,2): 160 窗口
    (P1-Staff1,1) × (P1-Staff2,0): 110 窗口
    (P1-Staff1,2) × (P1-Staff2,0): 100 窗口
    (P1-Staff2,5) × (P1-Staff2,6): 75 窗口
    (P1-Staff1,1) × (P1-Staff2,5): 64 窗口
    (P1-Staff1,1) × (P1-Staff2,6): 58 窗口
    (P1-Staff1,2) × (P1-Staff2,6): 41 窗口
    (P1-Staff1,2) × (P1-Staff2,5): 38 窗口
    (P1-Staff1,0) × (P1-Staff2,5): 36 窗口
    (P1-Staff1,0) × (P1-Staff2,6): 18 窗口
    (P1-Staff2,0) × (P1-Staff2,5): 3 窗口
    (P1-Staff2,0) × (P1-Staff2,6): 3 窗口
    (P1-Staff1,0) × (P1-Staff2,0): 2 窗口
✅ 窗口数据已保存: data/windows.pkl (1025 single + 708 mu